# Requirement 9: Protected communications

> Encrypt and authenticate sensitive traffic end to end between the trusted application and approved execution environment. Network location alone must not establish trust.

**Covered by:** Both

**How it is fulfilled:** Transport is TLS to an Entra-authenticated endpoint. Because the platform terminates TLS, the orchestrator adds application-layer encryption so data is protected end to end.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
import socket, ssl
host = f"management.{REGION}.azuredevcompute.io"
with socket.create_connection((host, 443), timeout=15) as raw:
    with ssl.create_default_context().wrap_socket(raw, server_hostname=host) as tls:
        print("TLS version:", tls.version())
        print("certificate issuer:", dict(x[0] for x in tls.getpeercert()["issuer"]).get("organizationName"))
        tls_version = tls.version()
check("transport is TLS 1.2 or newer with a validated certificate", tls_version in ("TLSv1.2", "TLSv1.3"))

TLS version: TLSv1.3
certificate issuer: Microsoft Corporation
PASS  transport is TLS 1.2 or newer with a validated certificate


In [3]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
sandbox, session_id = new_session()
session_key = install_session_key(sandbox, session_id)
secret = b"privileged: merger terms for Project Falcon"
nonce = os.urandom(12)
payload = nonce + AESGCM(session_key).encrypt(nonce, secret, session_id.encode())
check("the application-layer payload contains no plaintext", b"Falcon" not in payload)
sandbox.write_file("/tmp/payload.enc", payload)
check("only the session key holder can decrypt it", AESGCM(session_key).decrypt(nonce, payload[12:], session_id.encode()) == secret)
cleanup(sandbox)

PASS  the application-layer payload contains no plaintext
PASS  only the session key holder can decrypt it


**Caveat:** The app-layer encryption protects data on the way in and out, but anyone with data-plane access to the sandbox can read keys and memory. See requirements 1 and 10.